# HLK-ViT5

(HLK-ViT5 là tên do nhóm tự đặt cho model sau khi huấn luyện **ViT5-base** trên `8Opt/vietnamese-summarization-dataset-0001`)  

Notebook này dùng để chứa code để huấn luyện và dùng model sau khi huấn luyện  

Những cell code dùng để train đã được comment  

Mặc định hiện tại là sử dụng 300 mẫu để test model, nếu muốn tăng thì chỉnh sửa tham số `MAX_SAMPLES` ở mục 13  

Code sẽ mặc định bỏ qua các mẫu đã test, muốn cho chạy lại các mẫu đó thì cần xóa chúng đi hoặc chỉnh tham số `SKIP_EXISTING` ở mục 13


### 1. CẤU HÌNH DỰ ÁN LOCAL TRONG VS CODE
##### Code tạo đường dẫn bên dưới áp dụng cho VSCode, không chạy trên Google Colab
Tìm/Tạo/Kiểm tra đường dẫn đường dẫn


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()

# Đường dẫn mô hình base để fine-tune ở   models/ViT5-base/
LOCAL_MODEL = (PROJECT_ROOT / "models" / "ViT5-base")

# Đường dẫn lấy dataset ở   datasets/vietnamese-summarization-dataset-0001/
LOCAL_DATASET = (PROJECT_ROOT / "datasets" / "vietnamese-summarization-dataset-0001")

# Lưu checkpoint ở  training/vit5-base-hlk-0001-checkpoints/
CHECKPOINT_DIR = (PROJECT_ROOT / "training" / "vit5-base-hlk-0001-checkpoints")

# Mô hình sau khi fine-tune xong sẽ lưu ở   models/ViT5-base-HLK-0001/
FINAL_MODEL_DIR = (PROJECT_ROOT / "models" / "ViT5-base-HLK-0001")

# Kết quả mà mô hình tóm tắt sẽ lưu ở   results/vit5-base-HLK-0001/
RESULT_DIR = (PROJECT_ROOT / "results" / "vit5-base-HLK-0001")

# Tạo đường dẫn nếu chưa tồn tại
for directory in [CHECKPOINT_DIR, FINAL_MODEL_DIR, RESULT_DIR,]:
    directory.mkdir(parents=True, exist_ok=True,)

# Kiểm tra xem các đường dẫn có tồn tại không
assert LOCAL_MODEL.exists(), (f"Không tìm thấy model tại: {LOCAL_MODEL}")

assert LOCAL_DATASET.exists(), (f"Không tìm thấy dataset tại: {LOCAL_DATASET}")


# In ra các thông tin:
print("Python       :", sys.executable)
print("Project root :", PROJECT_ROOT)
print("Model gốc    :", LOCAL_MODEL)
print("Dataset      :", LOCAL_DATASET)
print("Checkpoint   :", CHECKPOINT_DIR)
print("Model cuối   :", FINAL_MODEL_DIR)
print("Kết quả      :", RESULT_DIR)


### 2. Cài đặt thư viện và model
Bỏ comment ra để chạy

In [ ]:
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
# %pip install -U transformers datasets accelerate evaluate rouge-score sentencepiece protobuf scikit-learn

In [ ]:
# Cài model đã được train fine-tune để chạy thử nghiệm luôn, không cần train lại

from huggingface_hub import snapshot_download

MODEL_ID = "tmanh217/hlk-vit5"

snapshot_download(
    repo_id=MODEL_ID,
    local_dir=FINAL_MODEL_DIR,
)

print(f"Đã tải model về: {FINAL_MODEL_DIR}")

### 3. Tải dataset LakoreAI/vietnamese-summarization-dataset-0001
Gỡ bỏ comment để tải dataset nếu chưa có trong folder datasets

In [ ]:
# from datasets import load_dataset

# dataset = load_dataset("LakoreAI/vietnamese-summarization-dataset-0001")

# for split in dataset.keys():
#     output_path = LOCAL_DATASET / f"{split}.jsonl"
#     dataset[split].to_json(output_path, force_ascii=False,)
#     print(f"Đã lưu {split}: {output_path}")

### 4. KIỂM TRA GPU VÀ LOAD DATASET

In [ ]:
import torch
from datasets import DatasetDict, load_dataset

print("CUDA:", torch.cuda.is_available())

# Kiểm tra xem có GPU không, nếu không có thì cảnh báo đang chạy bằng CPU.
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2,), "GB",)
else:
    print("Cảnh báo: đang chạy bằng CPU.")

# Load dataset
# Dảm bảo rằng các tệp train.jsonl, validation.jsonl, test.jsonl tồn tại trong thư mục datasets/vietnamese-summarization-dataset-0001/
data_files = {
    "train": str(LOCAL_DATASET / "train.jsonl"),  
    "validation": str(LOCAL_DATASET / "validation.jsonl"),
    "test": str(LOCAL_DATASET / "test.jsonl"),
}

# Load dataset từ các tệp JSONL
raw_dataset = load_dataset("json", data_files=data_files,)

print(raw_dataset)
print(
    "Các cột:",
    raw_dataset["train"].column_names,
)
print(
    "Mẫu đầu tiên:",
    raw_dataset["train"][0],
)

### 5. LÀM SẠCH, DEBUG VÀ QUẢN LÝ CHECKPOINT

In [ ]:
import shutil

TEXT_COLUMN = "document"
SUMMARY_COLUMN = "summary"
KEYWORDS_COLUMN = "keywords"

DEBUG_MODE = False
DEBUG_TRAIN_SIZE = 500
DEBUG_VALIDATION_SIZE = 100
DEBUG_TEST_SIZE = 100

# Chỉ đổi thành True khi chủ động train lại từ đầu.
RESET_CHECKPOINTS = False

# Kiểm tra 1 ví dụ có hợp lệ hay không để đưa vào mẫu
def valid_example(example):
    document = example.get(TEXT_COLUMN)
    summary = example.get(SUMMARY_COLUMN)
    return (isinstance(document, str) and isinstance(summary, str) and bool(document.strip()) and bool(summary.strip()))

clean_dataset = DatasetDict()

# Áp dụng hàm lọc valid_example cho từng tập của dataset
for split_name, split_dataset in raw_dataset.items():
    before = len(split_dataset)
    filtered = split_dataset.filter(valid_example, desc=f"Cleaning {split_name}",)
    clean_dataset[split_name] = filtered
    print(f"{split_name}: " + f"{len(filtered):,}/{before:,} mẫu hợp lệ")

raw_dataset = clean_dataset


# Chọn chế độ DEBUG để giảm kích thước dataset cho việc thử nghiệm nhanh
if DEBUG_MODE:
    dataset_for_training = DatasetDict({
        "train": raw_dataset["train"].select(range(min(DEBUG_TRAIN_SIZE, len(raw_dataset["train"]),))),
        "validation": raw_dataset["validation"].select(range(min(DEBUG_VALIDATION_SIZE, len(raw_dataset["validation"]),))),
        "test": raw_dataset["test"].select(range(min(DEBUG_TEST_SIZE, len(raw_dataset["test"]),))),
    })
    print("Đang chạy chế độ DEBUG.")
else:
    dataset_for_training = raw_dataset


# Xử lý checkpoint: nếu RESET_CHECKPOINTS = True thì xóa toàn bộ checkpoint hiện có, nếu False thì giữ nguyên và tiếp tục train từ checkpoint đã lưu.
if RESET_CHECKPOINTS:
    if CHECKPOINT_DIR.exists():
        shutil.rmtree(CHECKPOINT_DIR)

    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True,)
    print("Đã chủ động xóa checkpoint.")
else:
    checkpoint_names = sorted(
        path.name
        for path in CHECKPOINT_DIR.glob("checkpoint-*")
        if path.is_dir()
    )

    print("Giữ nguyên checkpoint hiện có.")
    print("Checkpoint:", checkpoint_names if checkpoint_names else "chưa có",)

print(dataset_for_training)


## Tiền xử lý long-context

Toàn bộ tài liệu được xét khi chọn thông tin:

```text
Văn bản dài
→ tách câu
→ chunk theo câu, có chồng lấn
→ chấm điểm bằng từ khóa + độ nổi bật + vị trí
→ MMR giảm chọn các chunk trùng nhau
→ sắp xếp Top-K chunk theo thứ tự gốc
→ ViT5 nhận tối đa 1.024 token
```


### 6. LOAD VIT5 VÀ CẤU HÌNH LONG-CONTEXT

In [ ]:
import ast
import json
import re
from collections import Counter

from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer,)

# ViT5 vẫn dùng full attention; 1.024 token tốn VRAM hơn 512.
MAX_INPUT_LENGTH = 1024
MAX_TARGET_LENGTH = 256
MAX_KEYWORD_TARGET_LENGTH = 64

# Tạo chunk theo câu trên toàn văn.
CHUNK_TOKEN_LENGTH = 256
CHUNK_OVERLAP_SENTENCES = 1
MAX_SELECTED_CHUNKS = 4

# Khoảng 20% mẫu bổ sung nhiệm vụ sinh từ khóa.
KEYWORD_TASK_EVERY = 4

# Trọng số chọn chunk.
KEYWORD_SCORE_WEIGHT = 0.45
SALIENCE_SCORE_WEIGHT = 0.35
POSITION_SCORE_WEIGHT = 0.20
MMR_REDUNDANCY_WEIGHT = 0.20

print("Đang load tokenizer...")

# Load tokenizer và model từ thư mục LOCAL_MODEL, đảm bảo chỉ sử dụng các tệp cục bộ.
tokenizer = AutoTokenizer.from_pretrained(str(LOCAL_MODEL), use_fast=False, local_files_only=True,)
print("Đang load model...")
model = AutoModelForSeq2SeqLM.from_pretrained(str(LOCAL_MODEL), local_files_only=True,)
model.config.use_cache = False
print("Số tham số:", f"{model.num_parameters():,}",)


VIETNAMESE_STOPWORDS = {
    "và", "là", "của", "có", "cho", "trong", "một",
    "những", "các", "được", "với", "để", "khi", "đã",
    "này", "đó", "từ", "theo", "về", "trên", "tại",
    "như", "do", "thì", "mà", "hay", "hoặc", "cũng",
    "không", "sẽ", "đang", "bị", "ra", "đến", "sau",
    "trước", "nhiều", "người", "việc", "lại", "nên",
}

# Chuẩn hóa văn bản: loại bỏ khoảng trắng thừa, chuẩn hóa xuống dòng, chuyển sang chuỗi.
def normalize_text(value) -> str:
    if value is None:
        return ""
    if isinstance(value, list):
        value = " ".join(str(item) for item in value)
    return " ".join(str(value).replace("\r\n", "\n").replace("\r", "\n").split()).strip()

# Tách từ trong văn bản, chỉ giữ lại các từ có ký tự chữ và số, bỏ dấu câu.
def word_tokens(text: str) -> list[str]:
    return re.findall(r"[0-9A-Za-zÀ-ỹĐđ]+", normalize_text(text).lower(),)

# Tách văn bản thành các câu dựa trên dấu chấm, chấm hỏi, chấm than, dấu ba chấm và xuống dòng.
def split_sentences(text: str) -> list[str]:
    text = str(text).replace("\r\n","\n",)
    parts = re.split(r"(?<=[.!?…])\s+|\n+", text,)
    sentences = [normalize_text(part) for part in parts if normalize_text(part)]
    return sentences or [normalize_text(text)]

# Tách từ khóa từ văn bản, chỉ giữ lại các từ có độ dài >= 3, không phải là stopword và không phải là số.
def derive_keywords(document: str, top_k: int = 8,):
    words = [word for word in word_tokens(document) if len(word) >= 3 and word not in VIETNAMESE_STOPWORDS and not word.isdigit()]
    counts = Counter(words)
    return [word for word, _ in counts.most_common(top_k)]

# Chuẩn hóa danh sách từ khóa.
def normalize_keywords(value, document: str, top_k: int = 10,):
    raw_items = []
    if isinstance(value, list):
        raw_items = value

    elif isinstance(value, str):
        stripped = value.strip()

        if stripped:
            parsed = None

            if (stripped.startswith("[") and stripped.endswith("]")):
                for parser in [json.loads, ast.literal_eval,]:
                    try:
                        parsed = parser(stripped)
                        break
                    except Exception:
                        pass

            if isinstance(parsed, list):
                raw_items = parsed
            else:
                raw_items = re.split(r"[;,\n|]+", stripped,)

    keywords = []
    seen = set()

    for item in raw_items:
        keyword = normalize_text(item)

        if not keyword:
            continue

        normalized = keyword.lower()

        if normalized in seen:
            continue

        seen.add(normalized)
        keywords.append(keyword)

        if len(keywords) >= top_k:
            break

    if not keywords:
        keywords = derive_keywords(document, top_k=min(top_k, 8),)
    return keywords

# Tách câu dài thành các đoạn nhỏ dựa trên số lượng token.
def split_long_sentence_by_tokens(sentence: str,):
    token_ids = tokenizer.encode(sentence, add_special_tokens=False,)
    if len(token_ids) <= CHUNK_TOKEN_LENGTH:
        return [sentence]
    pieces = []
    for start in range(0, len(token_ids), CHUNK_TOKEN_LENGTH):
        piece_ids = token_ids[start:start + CHUNK_TOKEN_LENGTH]
        piece = normalize_text(tokenizer.decode(piece_ids, skip_special_tokens=True,))
        if piece:
            pieces.append(piece)
    return pieces

# Xây dựng các chunk câu từ văn bản, đảm bảo mỗi chunk không vượt quá CHUNK_TOKEN_LENGTH token.
def build_sentence_chunks(document: str,):
    expanded_sentences = []
    for sentence in split_sentences(document):
        expanded_sentences.extend(split_long_sentence_by_tokens(sentence))

    chunks = []
    current_sentences = []
    current_tokens = 0

    for sentence in expanded_sentences:
        sentence_tokens = len(tokenizer.encode(sentence, add_special_tokens=False,))
        would_overflow = (current_sentences and current_tokens + sentence_tokens > CHUNK_TOKEN_LENGTH)
        if would_overflow:
            chunks.append(" ".join(current_sentences))
            overlap = (current_sentences[-CHUNK_OVERLAP_SENTENCES:] if CHUNK_OVERLAP_SENTENCES > 0 else [])
            current_sentences = list(overlap)
            current_tokens = sum(len(tokenizer.encode(item, add_special_tokens=False,)) for item in current_sentences)

        current_sentences.append(sentence)
        current_tokens += sentence_tokens

    if current_sentences:
        chunks.append(" ".join(current_sentences))
    return chunks

# Tính toán độ tương đồng Jaccard giữa hai văn bản dựa trên tập hợp từ.
def jaccard_similarity(text_a: str, text_b: str,) -> float:
    words_a = set(word_tokens(text_a))
    words_b = set(word_tokens(text_b))
    if not words_a or not words_b:
        return 0.0
    return len(words_a & words_b) / len(words_a | words_b)

# Chấm điểm và chọn các chunk từ văn bản dựa trên từ khóa, độ nổi bật và vị trí.
def score_and_select_chunks(document: str, keywords: list[str],) -> tuple[list[str], list[dict]]:
    chunks = build_sentence_chunks(document)

    if len(chunks) <= MAX_SELECTED_CHUNKS:
        metadata = [{"index": index, "score": 1.0,} for index in range(len(chunks))]
        return chunks, metadata

    document_words = [word for word in word_tokens(document) if word not in VIETNAMESE_STOPWORDS]

    frequencies = Counter(document_words)
    max_frequency = max(frequencies.values(), default=1,)

    base_scores = []

    for index, chunk in enumerate(chunks):
        chunk_lower = chunk.lower()
        chunk_words = [word for word in word_tokens(chunk) if word not in VIETNAMESE_STOPWORDS]
        keyword_coverage = (
            sum(keyword.lower() in chunk_lower for keyword in keywords) / max(len(keywords), 1))
        salience = (
            sum(frequencies[word] / max_frequency for word in chunk_words) / max(len(chunk_words), 1))
        position = (1.0 - index / max(len(chunks) - 1, 1))
        score = (KEYWORD_SCORE_WEIGHT * keyword_coverage + SALIENCE_SCORE_WEIGHT * salience + POSITION_SCORE_WEIGHT * position)
        base_scores.append(score)

    selected_indices = []
    remaining = set(range(len(chunks)))

    while (remaining and len(selected_indices) < MAX_SELECTED_CHUNKS):
        best_index = None
        best_mmr_score = -float("inf")

        for index in remaining:
            redundancy = max((jaccard_similarity(chunks[index], chunks[selected]) for selected in selected_indices), default=0.0,)
            mmr_score = (base_scores[index] - MMR_REDUNDANCY_WEIGHT * redundancy)
            if mmr_score > best_mmr_score:
                best_index = index
                best_mmr_score = mmr_score

        selected_indices.append(best_index)
        remaining.remove(best_index)

    # Giữ lại mạch văn ban đầu.
    selected_indices.sort()

    selected_chunks = [chunks[index] for index in selected_indices]

    metadata = [
        {
            "index": index,
            "score": round(base_scores[index], 6,),
        } for index in selected_indices]
    return selected_chunks, metadata

# Xây dựng input cho mô hình tóm tắt, bao gồm văn bản đã chọn, từ khóa và metadata.
def build_summary_input(document: str, keyword_value=None,) -> tuple[str, list[str], list[dict]]:
    document = normalize_text(document)
    keywords = normalize_keywords(keyword_value, document=document,)
    selected_chunks, chunk_metadata = (
        score_and_select_chunks(document, keywords,))
    selected_context = (" ĐOẠN TIẾP: ".join(selected_chunks))
    keyword_text = "; ".join(keywords)

    input_text = ("summarize: "
        f"keywords: {keyword_text} "
        f"document: {selected_context}"
    )

    return (input_text, keywords, chunk_metadata,)

### 7. TẠO DỮ LIỆU MULTI-TASK VÀ TOKENIZE

In [ ]:
# Tiền xử lý batch dữ liệu huấn luyện, bao gồm cả nhiệm vụ chính (tóm tắt) và nhiệm vụ phụ (sinh từ khóa).
def preprocess_train_batch(examples, indices,):
    documents = examples[TEXT_COLUMN]
    summaries = examples[SUMMARY_COLUMN]

    keyword_values = (examples[KEYWORDS_COLUMN] if KEYWORDS_COLUMN in examples else [None] * len(documents))

    input_texts = []
    target_texts = []

    for local_index, (document, summary, keyword_value,) in enumerate(
        zip(documents, summaries, keyword_values,)):
        summary_input, keywords, _ = (build_summary_input(document, keyword_value,))

        # Nhiệm vụ chính: tóm tắt.
        input_texts.append(summary_input)
        target_texts.append(normalize_text(summary))

        # Nhiệm vụ phụ: sinh từ khóa (~20%).
        global_index = indices[local_index]

        if (KEYWORD_TASK_EVERY > 0 and global_index % KEYWORD_TASK_EVERY == 0):
            keyword_context = (summary_input.split("document:", maxsplit=1,)[-1])
            input_texts.append("keywords: document: " + keyword_context)
            target_texts.append("; ".join(keywords))

    model_inputs = tokenizer(input_texts, max_length=MAX_INPUT_LENGTH, truncation=True,)
    labels = tokenizer(text_target=target_texts, max_length=MAX_TARGET_LENGTH, truncation=True,)
    model_inputs["labels"] = (labels["input_ids"])
    return model_inputs


# Tiền xử lý batch dữ liệu validation/test, chỉ bao gồm nhiệm vụ chính (tóm tắt).
def preprocess_summary_batch(examples):
    documents = examples[TEXT_COLUMN]
    summaries = examples[SUMMARY_COLUMN]

    keyword_values = (examples[KEYWORDS_COLUMN] if KEYWORDS_COLUMN in examples else [None] * len(documents))
    input_texts = []

    for document, keyword_value in zip(documents, keyword_values):
        summary_input, _, _ = (build_summary_input(document, keyword_value,))
        input_texts.append(summary_input)

    model_inputs = tokenizer(input_texts, max_length=MAX_INPUT_LENGTH, truncation=True,)
    labels = tokenizer(text_target=[normalize_text(summary) for summary in summaries], max_length=MAX_TARGET_LENGTH, truncation=True,)
    model_inputs["labels"] = (labels["input_ids"])
    return model_inputs


print("Đang tạo train multi-task...")

tokenized_train = (dataset_for_training["train"].map(preprocess_train_batch, batched=True, with_indices=True,
        remove_columns=dataset_for_training["train"].column_names, desc="Train: long-context + keyword task",))

print("Đang tokenize validation/test...")

tokenized_validation = (dataset_for_training["validation"].map(preprocess_summary_batch, batched=True, 
        remove_columns=dataset_for_training["validation"].column_names, desc="Validation: summary task",))

tokenized_test = (dataset_for_training["test"].map(preprocess_summary_batch, batched=True, 
        remove_columns=dataset_for_training["test"].column_names, desc="Test: summary task",))

tokenized_dataset = DatasetDict({
    "train": tokenized_train,
    "validation": tokenized_validation,
    "test": tokenized_test,
})

print(tokenized_dataset)
print("Train gốc:", len(dataset_for_training["train"]))
print("Train sau multi-task:", len(tokenized_dataset["train"]))

### 8. METRIC AN TOÀN, KHÔNG LỖI DECODE TOKEN ÂM

In [ ]:
import evaluate
import numpy as np

rouge_metric = evaluate.load("rouge")

# Hàm sanitize_token_ids đảm bảo rằng các token IDs hợp lệ trước khi giải mã, tránh lỗi decode token âm.
def sanitize_token_ids(values):
    values = np.asarray(values)

    if values.ndim == 3:
        values = np.argmax(values, axis=-1,)

    if np.issubdtype(values.dtype, np.floating,):
        values = np.nan_to_num(values, nan=tokenizer.pad_token_id, posinf=tokenizer.pad_token_id, neginf=tokenizer.pad_token_id,)

    values = values.astype(np.int64)
    valid_mask = ((values >= 0) & (values < len(tokenizer)))

    return np.where(valid_mask, values, tokenizer.pad_token_id,)


# Hàm compute_metrics tính toán các chỉ số ROUGE giữa dự đoán và nhãn thực tế, đồng thời tính độ dài trung bình của các dự đoán.
def compute_metrics(eval_prediction):
    predictions, labels = eval_prediction

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = sanitize_token_ids(predictions)
    labels = sanitize_token_ids(labels)
    decoded_predictions = (tokenizer.batch_decode(predictions, skip_special_tokens=True,))
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True,)
    decoded_predictions = [normalize_text(item) for item in decoded_predictions]
    decoded_labels = [normalize_text(item) for item in decoded_labels]
    result = rouge_metric.compute(predictions=decoded_predictions, references=decoded_labels, use_stemmer=False,)
    result["gen_len"] = float(np.mean([np.count_nonzero(prediction != tokenizer.pad_token_id) for prediction in predictions]))

    return {key: round(float(value), 4) for key, value in result.items()}


## Huấn luyện an toàn


### 9. TRAINER: ƯU TIÊN CHECKPOINT AN TOÀN

In [ ]:
import inspect
from transformers import (DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments,)

# 1.024 token cần batch nhỏ hơn.
# Batch hiệu dụng = 2 × 8 = 16.
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8

LEARNING_RATE = 3e-5    # Tốc độ học
NUM_TRAIN_EPOCHS = 3    # Số epoch huấn luyện

SAVE_STEPS = 100        # Lưu checkpoint sau mỗi 100 bước huấn luyện
EVAL_STEPS = 200        # Đánh giá sau mỗi 200 bước huấn luyện
LOGGING_STEPS = 20      # Ghi log sau mỗi 20 bước huấn luyện
SAVE_TOTAL_LIMIT = 3    # Giữ tối đa 3 checkpoint, xóa các checkpoint cũ hơn.

# Tạo data collator để xử lý batch dữ liệu cho mô hình seq2seq, đảm bảo padding và xử lý nhãn đúng cách.
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True, label_pad_token_id=-100, pad_to_multiple_of=8,)

# Tạo đối tượng Seq2SeqTrainingArguments với các tham số huấn luyện
# Bao gồm số epoch, tốc độ học, batch size, checkpoint, logging, và các tùy chọn khác.
def create_training_arguments():
    argument_names = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters

    use_bf16 = (torch.cuda.is_available() and hasattr(torch.cuda, "is_bf16_supported",) and torch.cuda.is_bf16_supported())

    kwargs = {
        "output_dir": str(CHECKPOINT_DIR),

        "num_train_epochs": NUM_TRAIN_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": 0.01,
        "warmup_ratio": 0.05,
        "lr_scheduler_type": "linear",

        "per_device_train_batch_size": TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": EVAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,

        "gradient_checkpointing": True,

        "fp16": (torch.cuda.is_available() and not use_bf16),
        "bf16": use_bf16,

        "logging_strategy": "steps",
        "logging_steps": LOGGING_STEPS,
        "logging_first_step": True,

        # save không bị chặn bởi compute_metrics.
        "save_strategy": "steps",
        "save_steps": SAVE_STEPS,
        "save_total_limit": SAVE_TOTAL_LIMIT,
        "save_only_model": False,
        "eval_steps": EVAL_STEPS,

        "predict_with_generate": True,
        "generation_max_length": MAX_TARGET_LENGTH,
        "generation_num_beams": 4,

        "load_best_model_at_end": False,

        "report_to": "none",
        "push_to_hub": False,

        "seed": 42,
        "data_seed": 42,

        "dataloader_num_workers": 0,
        "dataloader_pin_memory": False,
    }

    if "eval_strategy" in argument_names:
        kwargs["eval_strategy"] = "steps"
    else:
        kwargs["evaluation_strategy"] = "steps"

    return Seq2SeqTrainingArguments(**kwargs)

training_args = create_training_arguments()

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_dataset["train"],
    "eval_dataset": tokenized_dataset["validation"].select(range(50)),
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,}

trainer_parameters = inspect.signature(Seq2SeqTrainer.__init__).parameters

if "processing_class" in trainer_parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)
print(training_args)


### 10. TÌM CHECKPOINT HỢP LỆ MỚI NHẤT

In [ ]:
# # Tìm checkpoint hợp lệ mới nhất trong thư mục checkpoint, dựa trên các tệp cần thiết.
# def find_latest_valid_checkpoint(checkpoint_root: Path,):
#     valid_checkpoints = []
#     print("Đang tìm checkpoint tại:", checkpoint_root.resolve())

#     for path in checkpoint_root.glob("checkpoint-*"):
#         if not path.is_dir():
#             continue

#         try:
#             step = int(
#                 path.name.rsplit("-", 1)[-1])
#         except ValueError:
#             continue

#         model_files = ["model.safetensors", "model.safetensors.index.json", "pytorch_model.bin", "pytorch_model.bin.index.json",]
#         required_state_files = ["trainer_state.json", "optimizer.pt", "scheduler.pt",]
#         has_model = any((path / file_name).exists() for file_name in model_files)
#         missing_state_files = [file_name for file_name in required_state_files if not ( path / file_name).exists()]
#         print(path.name, "| model:", has_model, "| thiếu:", missing_state_files,)

#         if (has_model and not missing_state_files):
#             valid_checkpoints.append((step, path))

#     if not valid_checkpoints:
#         return None

#     valid_checkpoints.sort(key=lambda item: item[0])

#     return str(valid_checkpoints[-1][1])


# latest_checkpoint = (find_latest_valid_checkpoint(CHECKPOINT_DIR))

# if latest_checkpoint:
#     print("Sẽ tiếp tục từ:", latest_checkpoint,)
# else:
#     print("Không có checkpoint hợp lệ; " + "train từ model gốc.")

### 11. TRAIN

In [ ]:
# train_result = trainer.train(resume_from_checkpoint=latest_checkpoint)

# print("Train hoàn tất.")
# print(train_result.metrics)


### 12. LƯU MODEL CUỐI NGAY SAU KHI TRAIN XONG

In [ ]:
# model.config.use_cache = True
# trainer.save_model(str(FINAL_MODEL_DIR))
# tokenizer.save_pretrained(str(FINAL_MODEL_DIR))
# trainer.save_state()

# trainer.log_metrics("train", train_result.metrics)
# trainer.save_metrics("train", train_result.metrics,)

# pipeline_config = {
#     "max_input_length":
#         MAX_INPUT_LENGTH,
#     "max_target_length":
#         MAX_TARGET_LENGTH,
#     "chunk_token_length":
#         CHUNK_TOKEN_LENGTH,
#     "chunk_overlap_sentences":
#         CHUNK_OVERLAP_SENTENCES,
#     "max_selected_chunks":
#         MAX_SELECTED_CHUNKS,
#     "keyword_task_every":
#         KEYWORD_TASK_EVERY,
# }

# with (FINAL_MODEL_DIR / "hlk_pipeline_config.json").open("w", encoding="utf-8",) as file:
#     json.dump(pipeline_config, file, ensure_ascii=False, indent=2,)

# print("Đã lưu model:", FINAL_MODEL_DIR,)

### 13. ĐÁNH GIÁ SAU KHI MODEL ĐÃ ĐƯỢC LƯU

In [ ]:
# # Đặt None để đánh giá toàn bộ.
# # Nên thử 5 mẫu trước vì beam search khá chậm.
# EVALUATION_MAX_SAMPLES = 5

# evaluation_dataset = (tokenized_dataset["test"])
# original_test_dataset = (dataset_for_training["test"])

# if EVALUATION_MAX_SAMPLES is not None:
#     limit = min(EVALUATION_MAX_SAMPLES, len(evaluation_dataset),)
#     evaluation_dataset = (evaluation_dataset.select(range(limit)))
#     original_test_dataset = (original_test_dataset.select(range(limit)))

# print("Số mẫu đánh giá:", len(evaluation_dataset))
# test_result = trainer.predict(evaluation_dataset, metric_key_prefix="test",)
# trainer.log_metrics("test", test_result.metrics,)
# trainer.save_metrics("test", test_result.metrics,)
# print(test_result.metrics)

## Inference: sinh nhiều ứng viên và factuality reranking

Mỗi bài được xử lý bằng cùng cơ chế long-context như lúc train. ViT5 sinh nhiều ứng viên; bộ reranker ưu tiên:

- từ ngữ có căn cứ trong nguồn;
- bao phủ từ khóa;
- số liệu và thực thể xuất hiện trong nguồn;
- độ dài hợp lý;
- ít lặp.


### 14. BATCH TEST + FACTUALITY RERANKING
Mỗi bài được lưu ngay thành một file TXT.

In [ ]:
import time
from tqdm.auto import tqdm
from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer,)

# Load model
MODEL_DIR = FINAL_MODEL_DIR
TEST_FILE = LOCAL_DATASET / "test.jsonl"
OUTPUT_DIR = (RESULT_DIR / "test_predictions_txt")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True,)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer_test = (AutoTokenizer.from_pretrained(str(MODEL_DIR), use_fast=False, local_files_only=True,))
model_test = (AutoModelForSeq2SeqLM.from_pretrained(str(MODEL_DIR), local_files_only=True,))
model_test.to(device)
model_test.eval()



# Chỉnh sửa tham số ở đây
NUM_BEAMS = 4
NUM_RETURN_SEQUENCES = 4
MAX_NEW_TOKENS = 300
MIN_NEW_TOKENS = 30

SKIP_EXISTING = True        # True: bỏ qua các tệp đầu ra đã tồn tại, False: ghi đè.
MAX_SAMPLES = 300           # Hiện tại đang test trên 300 mẫu

# True: dùng keywords có sẵn trong JSONL.
# False: tự trích xuất keywords từ document.
USE_DATASET_KEYWORDS = False


# Hàm extract_capitalized_entities trích xuất các thực thể viết hoa từ văn bản, bao gồm cả các từ có dấu tiếng Việt.
def extract_capitalized_entities(text: str,) -> set[str]:
    entities = re.findall(r"\b(?:[A-ZÀ-ỸĐ][\wÀ-ỹĐđ.-]*" + r"(?:\s+[A-ZÀ-ỸĐ][\wÀ-ỹĐđ.-]*)+)\b", text,)
    return {normalize_text(entity).lower() for entity in entities if normalize_text(entity)}

# Hàm extract_numbers trích xuất các số từ văn bản, bao gồm cả các số thập phân và số có dấu phẩy.
def extract_numbers(text: str) -> set[str]:
    return set(re.findall(r"\b\d+(?:[.,]\d+)*\b", text,))


# Hàm repetition_ratio tính toán tỷ lệ lặp lại n-gram trong văn bản, giúp đánh giá mức độ lặp lại của văn bản.
def repetition_ratio(text: str, n: int = 3,) -> float:
    words = word_tokens(text)
    if len(words) < n:
        return 0.0

    ngrams = [tuple(words[index:index + n]) for index in range(len(words) - n + 1)]

    return (1.0 - len(set(ngrams)) / max(len(ngrams), 1))


# Hàm candidate_score tính toán các chỉ số đánh giá cho một bản tóm tắt ứng viên dựa trên văn bản gốc và từ khóa.
def candidate_score(summary: str, document: str, keywords: list[str],) -> dict:
    summary_words = set(word_tokens(summary))
    document_words = set(word_tokens(document))
    source_precision = (len(summary_words & document_words) / max(len(summary_words), 1))
    summary_lower = summary.lower()
    keyword_coverage = (sum(keyword.lower() in summary_lower for keyword in keywords) / max(len(keywords), 1))
    summary_numbers = extract_numbers(summary)
    document_numbers = extract_numbers(document)
    number_consistency = (1.0 if not summary_numbers else len(summary_numbers & document_numbers) / len(summary_numbers))
    summary_entities = (extract_capitalized_entities(summary))
    document_entities = (extract_capitalized_entities(document))
    entity_consistency = (1.0 if not summary_entities else len( summary_entities & document_entities) / len(summary_entities))
    input_length = max(len(word_tokens(document)), 1,)
    output_length = len(word_tokens(summary))
    compression_ratio = (output_length / input_length)

    # Tốt nhất quanh 10–25% độ dài đầu vào.
    length_score = max(0.0, 1.0 - abs(compression_ratio - 0.17) / 0.17,)
    repeat_ratio = repetition_ratio(summary)
    total_score = (0.35 * source_precision + 0.25 * keyword_coverage + 0.15 * number_consistency + 0.10 * entity_consistency
        + 0.10 * length_score + 0.05 * (1.0 - repeat_ratio))

    return {
        "score": total_score,
        "source_precision":
            source_precision,
        "keyword_coverage":
            keyword_coverage,
        "number_consistency":
            number_consistency,
        "entity_consistency":
            entity_consistency,
        "length_score": length_score,
        "repetition_ratio":
            repeat_ratio,
    }


# Hàm summarize_with_reranking thực hiện tóm tắt văn bản với việc xếp hạng lại các bản tóm tắt ứng viên dựa trên các chỉ số đánh giá.
@torch.inference_mode()
def summarize_with_reranking(document: str, keyword_value=None,):
    if not USE_DATASET_KEYWORDS:
        keyword_value = None

    input_text, keywords, chunk_metadata = (build_summary_input(document, keyword_value,))
    inputs = tokenizer_test(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LENGTH, padding=False,)
    inputs = {key: value.to(device) for key, value in inputs.items()}
    output_ids = model_test.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, min_new_tokens=MIN_NEW_TOKENS,
        num_beams=NUM_BEAMS, num_return_sequences= NUM_RETURN_SEQUENCES, do_sample=False, early_stopping=False,
        length_penalty=1.2, repetition_penalty=1.1, no_repeat_ngram_size=3,)

    candidates = tokenizer_test.batch_decode(output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True,)
    unique_candidates = []
    seen = set()

    for candidate in candidates:
        candidate = normalize_text(candidate)
        if (candidate and candidate not in seen):
            seen.add(candidate)
            unique_candidates.append(candidate)

    ranked_candidates = []

    for candidate in unique_candidates:
        details = candidate_score(candidate, document, keywords,)
        ranked_candidates.append({"summary": candidate, **details,})

    ranked_candidates.sort(key=lambda item: item["score"], reverse=True,)
    best = (ranked_candidates[0] if ranked_candidates else {"summary": "", "score": 0.0,})

    return {
        "final_summary":
            best["summary"],
        "best_score":
            best["score"],
        "keywords": keywords,
        "selected_chunks":
            chunk_metadata,
        "candidates":
            ranked_candidates,
        "input_tokens":
            int(
                inputs[
                    "input_ids"
                ].shape[1]
            ),
    }


# Hàm load_jsonl đọc tệp JSONL và trả về danh sách các bản ghi, bỏ qua các dòng không hợp lệ.
def load_jsonl(path: Path) -> list[dict]:
    records = []
    with path.open("r", encoding="utf-8",) as file:
        for line_number, line in enumerate(file, start=1,):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                print("Bỏ qua dòng", line_number, error,)
    return records


# Hàm safe_filename tạo tên tệp an toàn từ một chuỗi, loại bỏ các ký tự không hợp lệ và giới hạn độ dài.
def safe_filename(value: str, max_length: int = 70,) -> str:
    value = normalize_text(value)
    value = re.sub(r'[<>:"/\\|?*]', "_", value,)
    value = re.sub(r"\s+", "_", value,)
    value = value.strip("._ ")
    return (value[:max_length] if value else "untitled")


# Hàm save_result_txt lưu kết quả tóm tắt vào tệp văn bản
# Bao gồm thông tin về chỉ số, GUID, tiêu đề, văn bản gốc, tóm tắt tham chiếu, tóm tắt do mô hình sinh ra, thời gian thực hiện và điểm xếp hạng.
def save_result_txt(output_path: Path, index: int, guid: str, title: str, article: str, reference: str, generated: str,
    elapsed_seconds: float, rerank_score: float, keywords: list[str],):
    content = (
        "INDEX:\n"
        f"{index}\n"
        "GUID:\n"
        f"{guid}\n"
        "TITLE:\n"
        f"{title}\n"
        "VĂN BẢN GỐC:\n"
        f"{article}\n"
        "TÓM TẮT THAM CHIẾU:\n"
        f"{reference}\n"
        "TÓM TẮT DO MODEL SINH:\n"
        f"{generated}\n"
        "THỜI GIAN TÓM TẮT:\n"
        f"{elapsed_seconds:.4f} giây\n"
    )

    temporary_path = (output_path.with_suffix(".tmp"))

    with temporary_path.open("w", encoding="utf-8",) as file:
        file.write(content)
        file.flush()

    temporary_path.replace(output_path)

records = load_jsonl(TEST_FILE)

if MAX_SAMPLES is not None:
    records = records[:MAX_SAMPLES]

success_count = 0
skip_count = 0
error_count = 0


# Tiến hành tóm tắt và lưu kết quả cho từng bản ghi trong tập test, sử dụng tqdm để hiển thị tiến trình.
for index, record in enumerate(tqdm(records, desc="HLK-ViT5 test",)):
    guid = normalize_text(record.get("guid", record.get("id", index),))
    title = normalize_text(record.get("title", ""))
    article = normalize_text(record.get(TEXT_COLUMN, record.get("text", ""), ))
    reference = normalize_text(record.get(SUMMARY_COLUMN, "",))
    keyword_value = record.get(KEYWORDS_COLUMN)

    output_path = (
        OUTPUT_DIR / (
            f"{index:05d}"
            f"_guid-{safe_filename(guid, 25)}"
            f"_{safe_filename(title)}.txt"))

    if (SKIP_EXISTING and output_path.exists()):
        skip_count += 1
        continue

    try:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        started_at = time.perf_counter()
        result = summarize_with_reranking(article, keyword_value,)

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed = (time.perf_counter() - started_at)

        save_result_txt(
            output_path=output_path,
            index=index,
            guid=guid,
            title=title,
            article=article,
            reference=reference,
            generated=result["final_summary"],
            elapsed_seconds=elapsed,
            rerank_score=result["best_score"],
            keywords=result["keywords"],
        )
        success_count += 1

    except Exception as error:
        error_count += 1
        print(f"Lỗi bài {index}:", repr(error),)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("\nHoàn tất.")
print("Tạo mới :", success_count)
print("Bỏ qua  :", skip_count)
print("Bị lỗi  :", error_count)
print("Kết quả :", OUTPUT_DIR.resolve())

### 15. CHECKPOINT KHẨN CẤP
Chỉ chạy khi trainer.train() vừa lỗi nhưng kernel còn sống.

In [ ]:
# import inspect

# print("Global step:", trainer.state.global_step)
# save_function = (trainer._save_checkpoint)
# parameters = inspect.signature(save_function).parameters

# kwargs = {}

# if "model" in parameters:
#     kwargs["model"] = trainer.model

# if "trial" in parameters:
#     kwargs["trial"] = None

# save_function(**kwargs)

# print("Đã lưu checkpoint khẩn cấp tại:",(Path(trainer.args.output_dir) / ("checkpoint-" + str(trainer.state.global_step))),)